# Lab 3.4 - Build an End-to-End RAG Pipeline (Solution)
**Day 3 - Session 3**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/gen-ai-companion-intuite/blob/main/day3/lab3_4_rag/solution/lab3_4_rag.ipynb)

**Goal:** Build a retrieval-augmented generation pipeline over a small local
knowledge base. Chunk documents, embed them with sentence-transformers, index
with FAISS, retrieve the top chunks for a question, and answer with a grounded
prompt. Then score the answers with an LLM-as-a-judge.

This solution has the full working code. Retrieval runs offline; generation
and the judge call an LLM API through the shared client. Runs on CPU; no GPU
needed.

In [ ]:
# Scoped install for this lab (safe to re-run; preinstalled on Colab).
%pip install -q sentence-transformers faiss-cpu litellm python-dotenv

## Setup: make the shared client available
This finds the `shared/` package locally, or clones the companion repo when
running on Colab. It also makes sure an API key is set.

In [ ]:
import os
import subprocess
import sys


def _find_repo_root():
    d = os.getcwd()
    for _ in range(6):
        if os.path.isdir(os.path.join(d, "shared")):
            return d
        d = os.path.dirname(d)
    return None


root = _find_repo_root()
if root is None:  # On Colab: clone the companion repo for the shared client.
    if not os.path.isdir("gen-ai-companion-intuite"):
        subprocess.run(
            ["git", "clone", "-q",
             "https://github.com/kpassoubady/gen-ai-companion-intuite.git"],
            check=True,
        )
    root = "gen-ai-companion-intuite"
sys.path.insert(0, root)

from shared import get_completion, show_config

if not any(os.environ.get(k) for k in ("OPENAI_API_KEY", "GEMINI_API_KEY",
                                       "ANTHROPIC_API_KEY", "AZURE_API_KEY")):
    try:
        from google.colab import userdata
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    except Exception:
        from getpass import getpass
        os.environ["OPENAI_API_KEY"] = getpass("Enter your OPENAI_API_KEY: ")
    os.environ.setdefault("LLM_PROVIDER", "openai")

show_config()

## 1. A small knowledge base
Six short documents about a fictional product, "NimbusNote". These are the
only facts the assistant is allowed to use.

In [ ]:
DOCUMENTS = [
    {"source": "billing.md",
     "text": "NimbusNote offers three plans: Free, Pro at 8 dollars per month, "
             "and Team at 20 dollars per user per month. Annual billing saves "
             "16 percent. You can change or cancel your plan any time from "
             "Settings, Billing."},
    {"source": "storage.md",
     "text": "The Free plan includes 2 GB of storage. Pro includes 100 GB and "
             "Team includes 1 TB shared across the workspace. Attachments larger "
             "than 50 MB require a Pro or Team plan."},
    {"source": "sharing.md",
     "text": "Notes can be shared with a link or by inviting a teammate by "
             "email. Link sharing can be set to view-only or edit. Team plans "
             "support shared folders and per-folder permissions."},
    {"source": "security.md",
     "text": "NimbusNote encrypts data in transit and at rest. Two-factor "
             "authentication is available on all plans. Team admins can require "
             "two-factor for every member from the admin console."},
    {"source": "export.md",
     "text": "You can export a single note or your whole account as Markdown or "
             "PDF. Whole-account export is available on Pro and Team. Exports run "
             "in the background and you get an email when the file is ready."},
    {"source": "support.md",
     "text": "Free plan support is community forums only. Pro includes email "
             "support with a 48 hour response target. Team includes priority "
             "email support with a 12 hour response target."},
]
print(f"{len(DOCUMENTS)} documents loaded.")

## 2. Chunk the documents
Real documents are long, so we split them into overlapping chunks. These docs
are short, so each becomes one chunk, but the function is the pattern you use
at scale.

In [ ]:
def chunk_text(text, chunk_size=400, overlap=40):
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start:start + chunk_size])
        start += chunk_size - overlap
    return chunks


chunks = []
for doc in DOCUMENTS:
    for piece in chunk_text(doc["text"]):
        chunks.append({"source": doc["source"], "text": piece})
print(f"{len(chunks)} chunks after splitting.")

## 3. Embed and index with FAISS
We embed each chunk with a small local model and add the vectors to a FAISS
index. Normalizing the vectors makes inner-product search equal to cosine
similarity.

In [ ]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
chunk_texts = [c["text"] for c in chunks]
embeddings = embedder.encode(chunk_texts, normalize_embeddings=True)
embeddings = np.array(embeddings, dtype="float32")

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
print(f"Indexed {index.ntotal} chunks of dimension {embeddings.shape[1]}.")

## 4. Retrieve the top chunks
Embed the query the same way, search the index, and return the top-k chunks
with their similarity scores.

In [ ]:
def retrieve(query, k=3):
    q = embedder.encode([query], normalize_embeddings=True)
    scores, ids = index.search(np.array(q, dtype="float32"), k)
    hits = []
    for score, idx in zip(scores[0], ids[0]):
        hit = dict(chunks[idx])
        hit["score"] = float(score)
        hits.append(hit)
    return hits


for hit in retrieve("How much storage do I get on the Pro plan?"):
    print(f"  [{hit['score']:.2f}] {hit['source']}: {hit['text'][:70]}...")

## 5. Answer with a grounded prompt
We build a prompt that gives the model only the retrieved chunks and tells it
to answer from that context and cite sources. If the context lacks the answer,
it should say so instead of guessing.

In [ ]:
SYSTEM = (
    "You answer questions using ONLY the provided context. If the context does "
    "not contain the answer, say you do not have that information. Cite the "
    "source file in brackets, for example [billing.md]."
)


def rag_answer(query, k=3):
    hits = retrieve(query, k)
    context = "\n\n".join(f"[{h['source']}]\n{h['text']}" for h in hits)
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user",
         "content": f"Context:\n{context}\n\nQuestion: {query}"},
    ]
    return get_completion(messages, tier="mini", temperature=0.0), hits


answer, _ = rag_answer("How much storage does the Pro plan include?")
print(answer)

## 6. Ask a few questions
Grounded questions get grounded answers with citations. A question the KB
cannot answer should be declined instead of hallucinated.

In [ ]:
QUESTIONS = [
    "What is the monthly price of the Pro plan?",
    "How fast is support on the Team plan?",
    "Does NimbusNote have a mobile app?",  # not in the KB
]
for q in QUESTIONS:
    ans, _ = rag_answer(q)
    print(f"\nQ: {q}\nA: {ans}")

## 7. Evaluate with an LLM-as-a-judge
For a few golden question and answer pairs, we ask a judge model whether the
RAG answer is faithful to the retrieved context and whether it matches the
expected answer. The judge returns a small JSON we parse.

In [ ]:
import json

GOLDEN = [
    {"q": "How much does the Pro plan cost per month?", "expected": "8 dollars"},
    {"q": "How much storage does the Free plan include?", "expected": "2 GB"},
    {"q": "What is the Team support response target?", "expected": "12 hours"},
]

JUDGE_SYSTEM = (
    "You are a strict evaluator. Given a question, the retrieved context, the "
    "model answer, and the expected answer, return ONLY JSON like "
    '{"faithful": true, "correct": true} where faithful means the answer is '
    "supported by the context and correct means it matches the expected answer."
)


def judge(question, context, answer, expected):
    messages = [
        {"role": "system", "content": JUDGE_SYSTEM},
        {"role": "user", "content": (
            f"Question: {question}\nContext:\n{context}\n"
            f"Model answer: {answer}\nExpected: {expected}")},
    ]
    raw = get_completion(messages, tier="mini", temperature=0.0)
    raw = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```")
    return json.loads(raw)


passed = 0
for case in GOLDEN:
    ans, hits = rag_answer(case["q"])
    context = "\n\n".join(f"[{h['source']}]\n{h['text']}" for h in hits)
    verdict = judge(case["q"], context, ans, case["expected"])
    ok = verdict.get("faithful") and verdict.get("correct")
    passed += int(bool(ok))
    print(f"{'PASS' if ok else 'FAIL'}  {case['q']}  -> {verdict}")

print(f"\nScore: {passed}/{len(GOLDEN)} faithful and correct.")

## Stretch goals
1. **Agentic RAG.** Wrap `retrieve` and a small calculator as tools and give
   them to a smolagents `CodeAgent`, so the model decides when to search and
   when to compute (for example, the annual cost of the Team plan for 5 users).
2. **Swap the vector store.** Rebuild the index with Chroma
   (`chromadb.Client().create_collection(...)`) and confirm you get the same
   top chunks.
3. **Change chunking or the embedding model** (try `all-mpnet-base-v2`) and see
   whether retrieval scores and the judge results change.

In [ ]:
# Stretch 1: agentic RAG with smolagents (retrieve + compute as tools).
from smolagents import CodeAgent, LiteLLMModel, tool
from shared import get_model


@tool
def search_kb(query: str) -> str:
    """Search the NimbusNote knowledge base and return the top chunks.

    Args:
        query: The natural-language search query.
    """
    return "\n\n".join(f"[{h['source']}] {h['text']}" for h in retrieve(query, k=3))


@tool
def monthly_to_annual(monthly_cost: float, users: int) -> float:
    """Return the annual cost for a per-user monthly price.

    Args:
        monthly_cost: The monthly price per user in dollars.
        users: The number of users.
    """
    return round(monthly_cost * users * 12, 2)


agent = CodeAgent(tools=[search_kb, monthly_to_annual],
                  model=LiteLLMModel(model_id=get_model("mini")), max_steps=6)
print(agent.run("What is the annual cost of the Team plan for 5 users?"))

## Expected output
- Retrieval returns the storage chunk first for a storage question.
- Grounded answers cite the source file; the mobile-app question is declined
  because the fact is not in the knowledge base.
- The judge marks all three golden cases faithful and correct (3/3).
- The agentic stretch searches the KB for the Team price (20 dollars per user)
  and computes the annual cost for 5 users (1200 dollars).